# Colab Training: Balanced Data + SCL + EDAT

This notebook is designed to improve macro F1 for 8-class CWE classification by combining:

- Balanced sampling (class-aware oversampling)
- Class-weighted cross entropy with label smoothing
- Supervised Contrastive Loss (SCL)
- EDAT token augmentation in the training loop

It is self-contained and runs on Google Colab (GPU recommended).

In [ ]:
# Install dependencies (Colab)
!pip -q install transformers peft scikit-learn pyyaml

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import yaml
import numpy as np
import torch

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    BASE_DIR = Path('/content/drive/MyDrive/CSI_Project')
else:
    BASE_DIR = Path('/content') if Path('/content').exists() else Path.cwd()
    if not (BASE_DIR / 'config.yaml').exists():
        BASE_DIR = Path.cwd()

cfg_path = BASE_DIR / 'config.yaml'
assert cfg_path.exists(), f'Missing config.yaml at {cfg_path}'
with open(cfg_path, 'r') as f:
    base_cfg = yaml.safe_load(f)

cfg = {
    **base_cfg,
    'batch_size': 16,
    'epochs': 8,
    'learning_rate': 1.5e-5,
    'warmup_ratio': 0.1,
    'weight_decay': 0.01,
    'max_grad_norm': 1.0,
    'grad_accum_steps': 2,
    'label_smoothing': 0.05,
    'scl_temperature': 0.07,
    'scl_weight': 0.20,
    'edat_prob': 0.12,
    'edat_weight': 0.30,
    'early_stopping_patience': 4,
    'num_cwe_classes': 8,
    'seed': int(base_cfg.get('seed', 42)),
}

TOKEN_CACHE_DIR = BASE_DIR / cfg['token_cache_dir']
CHECKPOINT_DIR = BASE_DIR / cfg['checkpoint_dir']
LOG_DIR = BASE_DIR / cfg['log_dir']
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

MAX_LENGTH = int(cfg['max_length'])
CACHE_FILE = TOKEN_CACHE_DIR / f'tokens_maxlen{MAX_LENGTH}.pt'
assert CACHE_FILE.exists(), f'Missing cache: {CACHE_FILE}'

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

random.seed(cfg['seed'])
np.random.seed(cfg['seed'])
torch.manual_seed(cfg['seed'])
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(cfg['seed'])

print('BASE_DIR       :', BASE_DIR)
print('Device         :', DEVICE)
print('Cache          :', CACHE_FILE)
print('Checkpoint dir :', CHECKPOINT_DIR)
print('Epochs         :', cfg['epochs'])
print('Batch size     :', cfg['batch_size'])
print('Grad accum     :', cfg['grad_accum_steps'])

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

CWE_8_CLASSES = [
    'CWE-077',
    'CWE-601',
    'CWE-022',
    'CWE-094',
    'CWE-089',
    'CWE-352',
    'CWE-079',
    'unknown',
]

cache = torch.load(CACHE_FILE, weights_only=True)

class VulnerabilityDataset(Dataset):
    def __init__(self, cache_dict, split):
        if split == 'all':
            indices = list(range(len(cache_dict['split_origins'])))
        else:
            indices = [i for i, s in enumerate(cache_dict['split_origins']) if s == split]
        self.input_ids = cache_dict['input_ids'][indices]
        self.attention_mask = cache_dict['attention_mask'][indices]
        self.cwe_labels = cache_dict['cwe_labels'][indices]
        self.binary_labels = cache_dict['binary_labels'][indices]
        self.global_ids = cache_dict['global_ids'][indices]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            'input_ids': self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'cwe_label': self.cwe_labels[idx],
            'binary_label': self.binary_labels[idx],
            'global_id': self.global_ids[idx],
        }


def effective_number_weights(labels_tensor, num_classes=8, beta=0.999):
    counts = torch.bincount(labels_tensor, minlength=num_classes).float()
    eff_num = 1.0 - torch.pow(torch.tensor(beta), counts)
    weights = (1.0 - beta) / torch.clamp(eff_num, min=1e-8)
    weights = weights / weights.mean()
    return counts, weights


train_ds = VulnerabilityDataset(cache, 'train')
val_ds = VulnerabilityDataset(cache, 'val')

train_counts, class_weights = effective_number_weights(train_ds.cwe_labels, num_classes=8, beta=0.999)
sample_weights = class_weights[train_ds.cwe_labels.long()]
sampler = WeightedRandomSampler(
    weights=sample_weights.double(),
    num_samples=len(train_ds),
    replacement=True,
)

pin_mem = DEVICE == 'cuda'
num_workers = 2 if DEVICE != 'cpu' else 0

train_loader = DataLoader(
    train_ds,
    batch_size=cfg['batch_size'],
    sampler=sampler,
    num_workers=num_workers,
    pin_memory=pin_mem,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cfg['batch_size'],
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_mem,
)

print('Train samples:', len(train_ds), 'Val samples:', len(val_ds))
print('Train class distribution:')
for i, c in enumerate(train_counts.tolist()):
    pct = 100.0 * c / len(train_ds)
    print(f'  {i} {CWE_8_CLASSES[i]:<8} -> {int(c):>5} ({pct:5.2f}%)  weight={class_weights[i].item():.3f}')

In [ ]:
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model

tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])


class SupervisedContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        features = F.normalize(features, p=2, dim=1)
        labels = labels.view(-1, 1)

        sim = torch.matmul(features, features.T) / self.temperature
        sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

        self_mask = torch.eye(sim.size(0), device=sim.device, dtype=torch.bool)
        pos_mask = labels.eq(labels.T) & (~self_mask)

        exp_sim = torch.exp(sim) * (~self_mask)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-12)

        pos_count = pos_mask.sum(dim=1)
        valid = pos_count > 0
        if valid.sum() == 0:
            return torch.tensor(0.0, device=sim.device)

        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / pos_count.clamp(min=1)
        loss = -mean_log_prob_pos[valid].mean()
        return loss


class CWEContrastiveModel(nn.Module):
    def __init__(self, model_name, num_classes, class_weights, label_smoothing=0.05):
        super().__init__()
        encoder = AutoModel.from_pretrained(model_name)

        lora_cfg = LoraConfig(
            task_type=TaskType.FEATURE_EXTRACTION,
            r=int(cfg['lora_r']),
            lora_alpha=int(cfg['lora_alpha']),
            lora_dropout=float(cfg['lora_dropout']),
            target_modules=['query', 'value'],
            bias='none',
        )
        self.encoder = get_peft_model(encoder, lora_cfg)

        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_classes)
        self.projector = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden, 128),
        )

        self.ce_loss_fn = nn.CrossEntropyLoss(
            weight=class_weights,
            label_smoothing=label_smoothing,
        )

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls))
        proj = self.projector(cls)
        return {
            'logits': logits,
            'cls_repr': cls,
            'proj_repr': proj,
        }


model = CWEContrastiveModel(
    model_name=cfg['model_name'],
    num_classes=8,
    class_weights=class_weights.to(DEVICE),
    label_smoothing=float(cfg['label_smoothing']),
).to(DEVICE)

scl_loss_fn = SupervisedContrastiveLoss(temperature=float(cfg['scl_temperature']))

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,} ({100.0 * trainable / total:.2f}%)')

In [ ]:
# EDAT-style augmentation: random token masking on non-special tokens
SPECIAL_IDS = set(tokenizer.all_special_ids)
MASK_ID = tokenizer.mask_token_id if tokenizer.mask_token_id is not None else tokenizer.unk_token_id


def edat_augment(input_ids, attention_mask, mask_id, special_ids, prob=0.12):
    aug = input_ids.clone()

    special_mask = torch.zeros_like(aug, dtype=torch.bool)
    for sid in special_ids:
        special_mask |= aug.eq(sid)

    candidate = attention_mask.bool() & (~special_mask)
    rand = torch.rand_like(aug.float())
    replace = candidate & (rand < prob)
    aug[replace] = mask_id
    return aug

print('EDAT mask token id:', MASK_ID)

In [ ]:
# Small-batch sanity test for SCL + EDAT
batch = next(iter(train_loader))
input_ids = batch['input_ids'].to(DEVICE)
attn_mask = batch['attention_mask'].to(DEVICE)
labels = batch['cwe_label'].long().to(DEVICE)

with torch.no_grad():
    out_orig = model(input_ids, attn_mask)
    aug_ids = edat_augment(input_ids, attn_mask, MASK_ID, SPECIAL_IDS, prob=float(cfg['edat_prob']))
    out_aug = model(aug_ids, attn_mask)

    ce = model.ce_loss_fn(out_orig['logits'], labels)
    ce_aug = model.ce_loss_fn(out_aug['logits'], labels)

    feats = torch.cat([out_orig['proj_repr'], out_aug['proj_repr']], dim=0)
    labels_2 = torch.cat([labels, labels], dim=0)
    scl = scl_loss_fn(feats, labels_2)

    total = ce + float(cfg['edat_weight']) * ce_aug + float(cfg['scl_weight']) * scl

print('Small-batch test passed:')
print('  logits shape   :', tuple(out_orig['logits'].shape))
print('  proj shape     :', tuple(out_orig['proj_repr'].shape))
print(f'  CE             : {ce.item():.4f}')
print(f'  CE (EDAT)      : {ce_aug.item():.4f}')
print(f'  SCL            : {scl.item():.4f}')
print(f'  TOTAL          : {total.item():.4f}')

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from tqdm.auto import tqdm

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=float(cfg['learning_rate']),
    weight_decay=float(cfg['weight_decay']),
)

total_steps = (len(train_loader) // int(cfg['grad_accum_steps'])) * int(cfg['epochs'])
warmup_steps = int(total_steps * float(cfg['warmup_ratio']))
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=max(total_steps, 1),
)

BEST_CKPT = CHECKPOINT_DIR / 'best_scl_edat_model.pt'


def validate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    losses = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attn_mask = batch['attention_mask'].to(device)
            labels = batch['cwe_label'].long().to(device)

            out = model(input_ids, attn_mask)
            ce = model.ce_loss_fn(out['logits'], labels)

            preds = out['logits'].argmax(dim=-1)
            all_preds.extend(preds.detach().cpu().tolist())
            all_labels.extend(labels.detach().cpu().tolist())
            losses.append(ce.item())

    val_loss = float(np.mean(losses)) if losses else 0.0
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_p = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    macro_r = recall_score(all_labels, all_preds, average='macro', zero_division=0)

    return val_loss, macro_f1, macro_p, macro_r, all_preds, all_labels


def train_one_epoch(epoch_idx):
    model.train()
    running_loss = 0.0
    step_count = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch_idx}/{cfg["epochs"]}', leave=False)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar, start=1):
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['cwe_label'].long().to(DEVICE)

        out_orig = model(input_ids, attn_mask)
        aug_ids = edat_augment(input_ids, attn_mask, MASK_ID, SPECIAL_IDS, prob=float(cfg['edat_prob']))
        out_aug = model(aug_ids, attn_mask)

        ce = model.ce_loss_fn(out_orig['logits'], labels)
        ce_aug = model.ce_loss_fn(out_aug['logits'], labels)

        feats = torch.cat([out_orig['proj_repr'], out_aug['proj_repr']], dim=0)
        labels_2 = torch.cat([labels, labels], dim=0)
        scl = scl_loss_fn(feats, labels_2)

        loss = ce + float(cfg['edat_weight']) * ce_aug + float(cfg['scl_weight']) * scl
        loss = loss / int(cfg['grad_accum_steps'])
        loss.backward()

        if step % int(cfg['grad_accum_steps']) == 0:
            torch.nn.utils.clip_grad_norm_(
                filter(lambda p: p.requires_grad, model.parameters()),
                float(cfg['max_grad_norm']),
            )
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * int(cfg['grad_accum_steps'])
        step_count += 1

        if step % 20 == 0:
            pbar.set_postfix({'loss': f'{running_loss / max(step_count, 1):.4f}'})

    return running_loss / max(step_count, 1)

In [ ]:
history = []
best_f1 = -1.0
patience = int(cfg['early_stopping_patience'])
no_improve = 0

for epoch in range(1, int(cfg['epochs']) + 1):
    t0 = time.time()
    train_loss = train_one_epoch(epoch)

    val_loss, val_f1, val_p, val_r, val_preds, val_labels = validate(model, val_loader, DEVICE)
    dt = time.time() - t0

    row = {
        'epoch': epoch,
        'train_loss': round(float(train_loss), 4),
        'val_loss': round(float(val_loss), 4),
        'val_f1': round(float(val_f1), 4),
        'val_prec': round(float(val_p), 4),
        'val_rec': round(float(val_r), 4),
        'elapsed_s': round(float(dt), 1),
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
        f"val_F1={val_f1:.4f}  val_P={val_p:.4f}  val_R={val_r:.4f}  ({dt:.0f}s)"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        no_improve = 0
        torch.save({
            'epoch': epoch,
            'val_f1': float(val_f1),
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'config': cfg,
            'class_weights': class_weights.cpu(),
        }, BEST_CKPT)
        print('  New best checkpoint ->', BEST_CKPT)
    else:
        no_improve += 1
        print(f'  No improvement ({no_improve}/{patience})')
        if no_improve >= patience:
            print('Early stopping triggered.')
            break

print('Best validation macro F1:', round(best_f1, 4))

In [ ]:
# Load best checkpoint and print detailed validation report
ckpt = torch.load(BEST_CKPT, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

val_loss, val_f1, val_p, val_r, val_preds, val_labels = validate(model, val_loader, DEVICE)

print('Best checkpoint epoch :', ckpt['epoch'])
print('Validation macro F1   :', round(val_f1, 4))
print('Validation macro Prec :', round(val_p, 4))
print('Validation macro Rec  :', round(val_r, 4))
print()
print(classification_report(val_labels, val_preds, target_names=CWE_8_CLASSES, zero_division=0))

log = {
    'best_epoch': int(ckpt['epoch']),
    'best_val_f1': float(val_f1),
    'history': history,
    'config': cfg,
}
log_path = LOG_DIR / f"colab_scl_edat_run_{int(time.time())}.json"
with open(log_path, 'w') as f:
    json.dump(log, f, indent=2)

print('Saved log:', log_path)

## Quick Tuning Notes

If macro F1 is still low, tune in this order:

1. Increase `epochs` to 10-12 (keep early stopping).
2. Sweep `scl_weight` in [0.1, 0.2, 0.3].
3. Sweep `edat_prob` in [0.08, 0.12, 0.16].
4. Try `learning_rate` in [1.0e-5, 1.5e-5, 2.0e-5].
5. Keep both weighted sampler and weighted CE for imbalance.